# RL Challenge – Double-DQN-Baseline trainieren

Dieses Notebook klont immer die aktuelle Repository-Version, trainiert die einfache Double-DQN-Baseline als Spieler 1 gegen `baseline_pong_track_ball` und exportiert eine turnierkompatible Datei.

Vor dem Start: **Runtime → Change runtime type → T4 GPU** auswählen. Ohne GPU funktioniert das Notebook ebenfalls, aber deutlich langsamer.

In [ ]:
# Alte Kopie löschen: Colab-Speicher ist nur temporär und wird dadurch aktuell.
%cd /content
!rm -rf /content/rl-challenge
# Repository und alle benötigten Python-Pakete installieren.
!git clone --depth 1 https://github.com/georgmerz/rl-challenge.git
%cd rl-challenge
# Colab bringt PyTorch bereits mit; nur die kleineren Zusatzpakete installieren.
!pip install --prefer-binary -r requirements-colab.txt
# PettingZoo liefert die Atari-ROMs nicht mit. AutoROM installiert sie einmalig.
!AutoROM --accept-license

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch-Version: {torch.__version__}')
print(f'Verwendetes Gerät: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('Hinweis: Für schnelleres Training in Colab eine T4-GPU aktivieren.')

In [ ]:
# Parameter: Für einen ersten Test genügen 20_000 Schritte.
# 500_000 Schritte sind der normale längere Double-DQN-Trainingslauf.
# Alle 5.000 Schritte zeigt die deterministische ONNX-nahe Evaluation,
# ob das exportierte Modell wirklich gewinnt statt nur einen hohen Trainings-Reward zu haben.
TIMESTEPS = 500_000
OUTPUT = 'baselines/baseline_pong_ddqn.onnx'

!python train_ppo_sb3.py --device {device} --timesteps {TIMESTEPS} --eval-every 5000 --eval-episodes 5 --output {OUTPUT}

In [ ]:
# Der CPU-Validator entspricht der Modellprüfung im Turnier-Runner.
!python validate_submission.py {OUTPUT}

In [ ]:
# Die fertige, turnierkompatible Policy herunterladen.
from google.colab import files
files.download(OUTPUT)